### Main

In [1]:
from pathlib import Path
import routing_map
from routing_map import build_aoi, RoutingMapConfig
from routing_map.config import AoiConfig, LandConfig

cfg = RoutingMapConfig(
    aoi=AoiConfig(
        bbox_ll=(10, -50, 150, 50),
    ),
    land=LandConfig(
        shp_path=Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp"),
        buffer_km=20.0,
        avoid_km= 1,
        collision_safety_km=0.5,
    ),
)

out = build_aoi(cfg)

[coverage] attach gates->C: s_km assigned 926/926
[coverage] spacing sample: before=926 after=842 removed=84 (spacing_km=120.0, min_per_ring=1)


In [23]:
print("bbox_ll:", out["bbox_ll"])
print("bbox_ll_sea:", out.get("bbox_ll_sea"))
print("Gate_all_cov:", len(out.get("Gate_all_cov", [])))
print("S_nodes:", len(out.get("S_nodes", [])), "S_edges:", len(out.get("S_edges", [])))
print("gateB_connectors:", len(out.get("gateB_connectors", [])))
print("Gate_B_kept_gates:", len(out.get("Gate_B_kept_gates", [])))
print("sc_bundle_stats:", out.get("sc_bundle_stats"))


bbox_ll: (10, -10, 150, 50)
bbox_ll_sea: (7.0, -13.0, 153.0, 53.0)
Gate_all_cov: 714
S_nodes: 2579 S_edges: 4090
gateB_connectors: 221
Gate_B_kept_gates: 105
sc_bundle_stats: {'source': 'geograph_adjlist:latlon', 'node_count_parts': [2579], 'edge_count_parts': [4090], 'node_count': 2579, 'edge_count': 4090}


In [16]:
print(out.keys())

dict_keys(['cfg', 'bbox_ll', 'bbox_ll_sea', 'bbox_ll_sea_parts', 'proj', 'collision_prep', 'polys_ll', 'layers', 'union_smooth_m', 'ring_base_m', 'rings_m', 'rings_df', 'C_nodes', 'C_edges', 'F_nodes', 'F_att', 'Gate_A', 'Gate_F', 'Gate_all', 'Gate_all_aoi', 'Gate_all_cov', 'sc_bundle_stats', 'S_nodes', 'S_edges', 'sea_graph', 'sea_kdt', 'sea_ok_set', 'gateB_connectors', 'Gate_B_kept_gates'])


### Visualize Gate A, Gate F, Gate B

In [34]:
import folium
import webbrowser
from pathlib import Path
import numpy as np
import pandas as pd

def open_routing_debug_map(
    out: dict,
    *,
    bbox_ll=None,
    html_path="aoi_debug_map.html",
    zoom_start=5,
    # sampling / limits
    c_sample=8000,
    s_sample=None,            # sea nodes sample; None=all
    max_sea_edges=6000,       # avoid too many lines
):
    # ---------------- bbox ----------------
    if bbox_ll is None:
        bbox_ll = out.get("bbox_ll", None)
    if bbox_ll is None:
        raise ValueError("bbox_ll is required (pass bbox_ll=... or ensure out['bbox_ll'] exists).")

    min_lon, min_lat, max_lon, max_lat = [float(x) for x in bbox_ll]
    center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

    m = folium.Map(location=center, zoom_start=zoom_start, control_scale=True)

    # ---------------- color theme ----------------
    COLORS = {
        "bbox": "#A020F0",        # purple
        "C": "#1f77b4",           # blue
        "Gate_all": "#111111",    # near-black
        "Gate_A": "#d62728",      # red
        "Gate_F": "#ff7f0e",      # orange
        "Gate_B": "#6f42c1",      # violet
        "Sea": "#17becf",         # cyan (sea nodes + sea edges)
        "Conn": "#2ca02c",        # green (GateB->Sea connectors)
    }

    # bbox rectangle
    folium.Rectangle(
        bounds=[[min_lat, min_lon], [max_lat, max_lon]],
        fill=False, weight=3, opacity=0.9, color=COLORS["bbox"]
    ).add_to(m)
    m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    # ---------------- helpers ----------------
    def _safe_df(name):
        df = out.get(name, None)
        if df is None:
            return None
        try:
            return df if len(df) > 0 else None
        except Exception:
            return None

    def _circle_layer(df, *, layer_name, radius, color, show=True, fill_opacity=0.75, line_opacity=0.9):
        fg = folium.FeatureGroup(name=layer_name, show=show)
        for _, r in df.iterrows():
            folium.CircleMarker(
                [float(r["lat"]), float(r["lon"])],
                radius=radius,
                color=color,
                weight=1,
                opacity=line_opacity,
                fill=True,
                fill_color=color,
                fill_opacity=fill_opacity,
            ).add_to(fg)
        fg.add_to(m)

    # gate lookup (g_id -> lon/lat)
    def _build_gate_xy():
        gate_df = _safe_df("Gate_all_cov")
        if gate_df is None:
            gate_df = _safe_df("Gate_all")
        if gate_df is None or ("g_id" not in gate_df.columns):
            return {}
        return {
            int(r["g_id"]): (float(r["lon"]), float(r["lat"]))
            for _, r in gate_df.iterrows()
        }

    gate_xy = _build_gate_xy()

    # sea node lookup by idx
    S_nodes = _safe_df("S_nodes")

    def _sea_lonlat_by_idx(i):
        s = S_nodes.iloc[int(i)]
        return float(s["lon"]), float(s["lat"])

    # sea edge to lonlat
    def _parse_node_id_str(s):
        try:
            a, b = s.split(",")
            return float(a), float(b)
        except Exception:
            return None

    def _edge_to_lonlat(e):
        """
        Return ((lon1,lat1),(lon2,lat2)) or None.
        Supports:
          1) (u_idx, v_idx) using S_nodes
          2) ((lon,lat),(lon,lat))
          3) ("lon,lat","lon,lat")
        """
        if not isinstance(e, (list, tuple)) or len(e) < 2:
            return None
        a, b = e[0], e[1]

        # (u_idx, v_idx)
        if isinstance(a, (int, np.integer)) and isinstance(b, (int, np.integer)):
            if S_nodes is None:
                return None
            try:
                p1 = _sea_lonlat_by_idx(a)
                p2 = _sea_lonlat_by_idx(b)
                return (p1, p2)
            except Exception:
                return None

        # ((lon,lat),(lon,lat))
        if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)) and len(a) >= 2 and len(b) >= 2:
            try:
                return ((float(a[0]), float(a[1])), (float(b[0]), float(b[1])))
            except Exception:
                return None

        # ("lon,lat","lon,lat")
        if isinstance(a, str) and isinstance(b, str):
            p1 = _parse_node_id_str(a)
            p2 = _parse_node_id_str(b)
            if p1 and p2:
                return (p1, p2)

        return None

    # ---------------- layers ----------------
    # C nodes
    dfC = _safe_df("C_nodes")
    if dfC is not None:
        nC = len(dfC)
        dfC_plot = dfC.sample(min(int(c_sample), nC), random_state=7) if nC > c_sample else dfC
        _circle_layer(
            dfC_plot,
            layer_name=f"C_nodes (sample {len(dfC_plot)}/{nC})",
            radius=1,
            color=COLORS["C"],
            show=True,
            fill_opacity=0.65,
            line_opacity=0.85,
        )

    # Gates: all / A / F
    # Gate_all_cov / Gate_all 用同一色（黑）
    for k, show, color in [
        ("Gate_all_cov", True, COLORS["Gate_all"]),
        ("Gate_all", True, COLORS["Gate_all"]),
        ("Gate_A", False, COLORS["Gate_A"]),
        ("Gate_F", False, COLORS["Gate_F"]),
    ]:
        dfG = _safe_df(k)
        if dfG is not None:
            _circle_layer(
                dfG,
                layer_name=f"{k} ({len(dfG)})",
                radius=4 if "Gate_all" in k else 5,
                color=color,
                show=show,
                fill_opacity=0.8,
                line_opacity=0.95,
            )

    # Gate B (kept gates)
    gb_obj = out.get("Gate_B_kept_gates", None)
    dfGB = None

    if gb_obj is not None:
        if isinstance(gb_obj, pd.DataFrame):
            dfGB = gb_obj.copy()
            if "lon" not in dfGB.columns or "lat" not in dfGB.columns:
                if "g_id" in dfGB.columns and gate_xy:
                    dfGB["lon"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[0])
                    dfGB["lat"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[1])
            dfGB = dfGB.dropna(subset=["lon", "lat"])
        elif isinstance(gb_obj, (list, set, tuple, np.ndarray)):
            gids = [int(x) for x in gb_obj]
            rows = []
            for gid in gids:
                if gid in gate_xy:
                    lon, lat = gate_xy[gid]
                    rows.append({"g_id": gid, "lon": lon, "lat": lat})
            dfGB = pd.DataFrame(rows)

    if dfGB is None:
        gb2 = _safe_df("Gate_B")
        if gb2 is not None:
            dfGB = gb2.copy()

    if dfGB is not None and len(dfGB) > 0:
        _circle_layer(
            dfGB,
            layer_name=f"Gate_B ({len(dfGB)})",
            radius=6,
            color=COLORS["Gate_B"],
            show=True,
            fill_opacity=0.85,
            line_opacity=0.95,
        )

    # Sea nodes
    if S_nodes is not None:
        nS = len(S_nodes)
        if s_sample is not None and nS > int(s_sample):
            dfS_plot = S_nodes.sample(int(s_sample), random_state=7)
            title = f"S_nodes (sample {len(dfS_plot)}/{nS})"
        else:
            dfS_plot = S_nodes
            title = f"S_nodes ({nS})"

        _circle_layer(
            dfS_plot,
            layer_name=title,
            radius=3,
            color=COLORS["Sea"],
            show=True,
            fill_opacity=0.75,
            line_opacity=0.9,
        )

    # Sea edges (same color as sea nodes)
    S_edges = out.get("S_edges", None)
    if S_nodes is not None and S_edges is not None and len(S_edges) > 0:
        fgE = folium.FeatureGroup(
            name=f"S_edges (show {min(len(S_edges), max_sea_edges)}/{len(S_edges)})",
            show=True
        )

        take = S_edges[:max_sea_edges] if len(S_edges) > max_sea_edges else S_edges
        drawn = 0
        for e in take:
            seg = _edge_to_lonlat(e)
            if seg is None:
                continue
            (lon1, lat1), (lon2, lat2) = seg
            folium.PolyLine(
                [[lat1, lon1], [lat2, lon2]],
                weight=2,
                opacity=0.7,
                color=COLORS["Sea"],
            ).add_to(fgE)
            drawn += 1

        fgE.add_to(m)
        print(f"[viz] sea edges drawn: {drawn}/{len(take)}")

    # GateB -> sea connectors (green)
    dfConn = _safe_df("gateB_connectors")
    if dfConn is not None and S_nodes is not None and gate_xy:
        fgConn = folium.FeatureGroup(name=f"GateB→Sea connectors ({len(dfConn)})", show=True)

        kept_gids = None
        if dfGB is not None and "g_id" in dfGB.columns and len(dfGB) > 0:
            kept_gids = set(int(x) for x in dfGB["g_id"].values)

        drawn = 0
        for _, r in dfConn.iterrows():
            gid = int(r["g_id"])
            if kept_gids is not None and gid not in kept_gids:
                continue
            if gid not in gate_xy:
                continue
            lon_g, lat_g = gate_xy[gid]
            try:
                lon_s, lat_s = _sea_lonlat_by_idx(int(r["sea_idx"]))
            except Exception:
                continue

            folium.PolyLine(
                [[lat_g, lon_g], [lat_s, lon_s]],
                weight=2,
                opacity=0.95,
                color=COLORS["Conn"],
            ).add_to(fgConn)
            drawn += 1

        fgConn.add_to(m)
        print(f"[viz] GateB→Sea connectors drawn: {drawn}/{len(dfConn)}")

    folium.LayerControl(collapsed=False).add_to(m)

    html_path = Path(html_path).resolve()
    m.save(str(html_path))
    webbrowser.open(html_path.as_uri())
    return html_path


In [35]:
html = open_routing_debug_map(out, bbox_ll=out["bbox_ll"], html_path="aoi_debug_map.html")
print("saved:", html)


[viz] sea edges drawn: 1536/1536
[viz] GateB→Sea connectors drawn: 157/157
saved: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\aoi_debug_map.html


### Phase I 驗收

In [8]:
import folium
import webbrowser
from pathlib import Path
import numpy as np
import pandas as pd

def open_routing_debug_map(
    out: dict,
    *,
    bbox_ll=None,
    html_path="aoi_debug_map.html",
    zoom_start=5,
    # sampling / limits
    c_sample=8000,
    s_sample=None,            # sea nodes sample; None=all
    max_sea_edges=6000,       # avoid too many lines
):
    # ---------------- bbox ----------------
    if bbox_ll is None:
        bbox_ll = out.get("bbox_ll", None)
    if bbox_ll is None:
        raise ValueError("bbox_ll is required (pass bbox_ll=... or ensure out['bbox_ll'] exists).")

    min_lon, min_lat, max_lon, max_lat = [float(x) for x in bbox_ll]
    center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

    m = folium.Map(location=center, zoom_start=zoom_start, control_scale=True)

    # bbox rectangle
    folium.Rectangle(
        bounds=[[min_lat, min_lon], [max_lat, max_lon]],
        fill=False, weight=3, opacity=0.9
    ).add_to(m)
    m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    # ---------------- helpers ----------------
    def _safe_df(name):
        df = out.get(name, None)
        if df is None:
            return None
        try:
            return df if len(df) > 0 else None
        except Exception:
            return None

    def _circle_layer(df, *, layer_name, radius, show=True):
        fg = folium.FeatureGroup(name=layer_name, show=show)
        for _, r in df.iterrows():
            folium.CircleMarker([float(r["lat"]), float(r["lon"])], radius=radius).add_to(fg)
        fg.add_to(m)

    # gate lookup (g_id -> lon/lat)
    def _build_gate_xy():
        # coverage 後優先：不要用 `or`，DataFrame 不能做 truthy 判斷
        gate_df = _safe_df("Gate_all_cov")
        if gate_df is None:
            gate_df = _safe_df("Gate_all")

        if gate_df is None or ("g_id" not in gate_df.columns):
            return {}

        return {
            int(r["g_id"]): (float(r["lon"]), float(r["lat"]))
            for _, r in gate_df.iterrows()
        }

    gate_xy = _build_gate_xy()

    # sea node lookup by idx
    S_nodes = _safe_df("S_nodes")

    def _sea_lonlat_by_idx(i):
        s = S_nodes.iloc[int(i)]
        return float(s["lon"]), float(s["lat"])

    # sea edge to lonlat
    def _parse_node_id_str(s):
        try:
            a, b = s.split(",")
            return float(a), float(b)
        except Exception:
            return None

    def _edge_to_lonlat(e):
        """
        Return ((lon1,lat1),(lon2,lat2)) or None.
        Supports:
          1) (u_idx, v_idx) using S_nodes
          2) ((lon,lat),(lon,lat))
          3) ("lon,lat","lon,lat")
        """
        if not isinstance(e, (list, tuple)) or len(e) < 2:
            return None
        a, b = e[0], e[1]

        # (u_idx, v_idx)
        if isinstance(a, (int, np.integer)) and isinstance(b, (int, np.integer)):
            if S_nodes is None:
                return None
            try:
                p1 = _sea_lonlat_by_idx(a)
                p2 = _sea_lonlat_by_idx(b)
                return (p1, p2)
            except Exception:
                return None

        # ((lon,lat),(lon,lat))
        if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)) and len(a) >= 2 and len(b) >= 2:
            try:
                return ((float(a[0]), float(a[1])), (float(b[0]), float(b[1])))
            except Exception:
                return None

        # ("lon,lat","lon,lat")
        if isinstance(a, str) and isinstance(b, str):
            p1 = _parse_node_id_str(a)
            p2 = _parse_node_id_str(b)
            if p1 and p2:
                return (p1, p2)

        return None

    # ---------------- layers ----------------
    # C nodes
    dfC = _safe_df("C_nodes")
    if dfC is not None:
        nC = len(dfC)
        dfC_plot = dfC.sample(min(int(c_sample), nC), random_state=7) if nC > c_sample else dfC
        _circle_layer(dfC_plot, layer_name=f"C_nodes (sample {len(dfC_plot)}/{nC})", radius=1, show=True)

    # Gates: all / A / F
    for k, show in [("Gate_all_cov", True), ("Gate_all", True), ("Gate_A", False), ("Gate_F", False)]:
        dfG = _safe_df(k)
        if dfG is not None:
            _circle_layer(dfG, layer_name=f"{k} ({len(dfG)})", radius=4, show=show)

    # Gate B (kept gates)
    gb_obj = out.get("Gate_B_kept_gates", None)
    dfGB = None

    if gb_obj is not None:
        if isinstance(gb_obj, pd.DataFrame):
            dfGB = gb_obj.copy()
            if "lon" not in dfGB.columns or "lat" not in dfGB.columns:
                if "g_id" in dfGB.columns and gate_xy:
                    dfGB["lon"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[0])
                    dfGB["lat"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[1])
            dfGB = dfGB.dropna(subset=["lon", "lat"])
        elif isinstance(gb_obj, (list, set, tuple, np.ndarray)):
            gids = [int(x) for x in gb_obj]
            rows = []
            for gid in gids:
                if gid in gate_xy:
                    lon, lat = gate_xy[gid]
                    rows.append({"g_id": gid, "lon": lon, "lat": lat})
            dfGB = pd.DataFrame(rows)

    # fallback: maybe out["Gate_B"] exists
    if dfGB is None:
        gb2 = _safe_df("Gate_B")
        if gb2 is not None:
            dfGB = gb2.copy()

    if dfGB is not None and len(dfGB) > 0:
        _circle_layer(dfGB, layer_name=f"Gate_B ({len(dfGB)})", radius=6, show=True)

    # Sea nodes
    if S_nodes is not None:
        nS = len(S_nodes)
        if s_sample is not None and nS > int(s_sample):
            dfS_plot = S_nodes.sample(int(s_sample), random_state=7)
            title = f"S_nodes (sample {len(dfS_plot)}/{nS})"
        else:
            dfS_plot = S_nodes
            title = f"S_nodes ({nS})"
        _circle_layer(dfS_plot, layer_name=title, radius=3, show=True)

    # Sea edges
    S_edges = out.get("S_edges", None)
    if S_nodes is not None and S_edges is not None and len(S_edges) > 0:
        fgE = folium.FeatureGroup(name=f"S_edges (show {min(len(S_edges), max_sea_edges)}/{len(S_edges)})", show=True)

        take = S_edges[:max_sea_edges] if len(S_edges) > max_sea_edges else S_edges
        drawn = 0
        for e in take:
            seg = _edge_to_lonlat(e)
            if seg is None:
                continue
            (lon1, lat1), (lon2, lat2) = seg
            folium.PolyLine([[lat1, lon1], [lat2, lon2]], weight=2, opacity=0.8).add_to(fgE)
            drawn += 1

        fgE.add_to(m)
        print(f"[viz] sea edges drawn: {drawn}/{len(take)}")

    # GateB -> sea connectors
    dfConn = _safe_df("gateB_connectors")
    if dfConn is not None and S_nodes is not None and gate_xy:
        fgConn = folium.FeatureGroup(name=f"GateB→Sea connectors ({len(dfConn)})", show=True)

        kept_gids = None
        if dfGB is not None and "g_id" in dfGB.columns and len(dfGB) > 0:
            kept_gids = set(int(x) for x in dfGB["g_id"].values)

        drawn = 0
        for _, r in dfConn.iterrows():
            gid = int(r["g_id"])
            if kept_gids is not None and gid not in kept_gids:
                continue
            if gid not in gate_xy:
                continue
            lon_g, lat_g = gate_xy[gid]
            try:
                lon_s, lat_s = _sea_lonlat_by_idx(int(r["sea_idx"]))
            except Exception:
                continue

            folium.PolyLine([[lat_g, lon_g], [lat_s, lon_s]], weight=2, opacity=0.95).add_to(fgConn)
            drawn += 1

        fgConn.add_to(m)
        print(f"[viz] GateB→Sea connectors drawn: {drawn}/{len(dfConn)}")

    folium.LayerControl(collapsed=False).add_to(m)

    html_path = Path(html_path).resolve()
    m.save(str(html_path))
    webbrowser.open(html_path.as_uri())
    return html_path


#### 畫圖

In [70]:
html = open_routing_debug_map(out, bbox_ll=out["bbox_ll"], html_path="aoi_debug_map.html")
print("saved:", html)


[viz] sea edges drawn: 390/390
[viz] GateB→Sea connectors drawn: 54/54
saved: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\aoi_debug_map.html


#### 選定一條劃出

In [112]:
import networkx as nx
import numpy as np
import pandas as pd

def test_gate_navigation_v8_final_fixed(out):
    G = nx.Graph()
    
    # 1. 解析海洋路網 (S_edges) - 節點是 tuple
    sea_edges = out.get('S_edges', [])
    for u, v in sea_edges:
        dist = np.sqrt((u[0]-v[0])**2 + (u[1]-v[1])**2)
        G.add_edge(u, v, weight=dist, type='sea')

    # 2. 解析 Gate_B 座標 (建立 g_id 到 座標的對照)
    gates_df = out.get('Gate_B_kept_gates')
    gate_id_to_ll = {}
    if isinstance(gates_df, pd.DataFrame):
        for _, row in gates_df.iterrows():
            # 根據你提供的欄位，g_id 是關聯鍵
            ll = (row['lon'], row['lat'])
            gate_id_to_ll[row['g_id']] = ll

    # 3. 解析 gateB_connectors (執行「焊接」)
    conn_df = out.get('gateB_connectors')
    if isinstance(conn_df, pd.DataFrame):
        for _, row in conn_df.iterrows():
            # A. 取得 Gate 座標
            g_ll = gate_id_to_ll.get(row['g_id'])
            
            # B. 取得 Sea Node 座標
            # 處理字串格式 "135.456200,42.237800"
            s_raw = row['sea_node_id']
            try:
                if isinstance(s_raw, str):
                    s_ll = tuple(map(float, s_raw.split(',')))
                else:
                    s_ll = s_raw
                
                if g_ll and s_ll:
                    # 計算距離並連線
                    dist = np.sqrt((g_ll[0]-s_ll[0])**2 + (g_ll[1]-s_ll[1])**2)
                    G.add_edge(g_ll, s_ll, weight=dist, type='connector')
                    G.nodes[g_ll]['type'] = 'gate'
            except:
                continue

    # 4. 統計與測試
    valid_gates = [ll for ll in gate_id_to_ll.values() if ll in G]
    print(f" 最終結果：節點={G.number_of_nodes()}, 邊={G.number_of_edges()}")
    print(f" 成功焊接進圖中的 Gate 數量: {len(valid_gates)}")

    if len(valid_gates) < 2:
        print(" 焊接失敗，請確認 Gate 與 Sea Node 的座標格式是否完全一致。")
        return None

    # 5. A* 導航
    try:
        start_node, end_node = valid_gates[0], valid_gates[-1]
        path = nx.astar_path(G, start_node, end_node, 
                             heuristic=lambda n1, n2: np.sqrt((n1[0]-n2[0])**2 + (n1[1]-n2[1])**2), 
                             weight='weight')
        
        edges_in_path = list(zip(path, path[1:]))
        conn_used = sum(1 for u, v in edges_in_path if G[u][v].get('type') == 'connector')
        
        print(f" 導航成功！路徑點數: {len(path)}, 跨越 Connector 次數: {conn_used}")
        return path
    except Exception as e:
        print(f" 導航失敗: {e}")
        return None

path = test_gate_navigation_v8_final_fixed(out)

 最終結果：節點=1967, 邊=3168
 成功焊接進圖中的 Gate 數量: 24
 導航成功！路徑點數: 43, 跨越 Connector 次數: 2


#### 隨機兩個gateB 互相導航

In [113]:
import networkx as nx
import numpy as np
import pandas as pd

def test_gate_navigation_random_pair(out, seed=None):
    G = nx.Graph()

    # 1) Sea edges
    sea_edges = out.get('S_edges', [])
    for u, v in sea_edges:
        dist = np.sqrt((u[0]-v[0])**2 + (u[1]-v[1])**2)
        G.add_edge(u, v, weight=dist, type='sea')

    # 2) Gate_B id -> ll
    gates_df = out.get('Gate_B_kept_gates')
    gate_id_to_ll = {}
    ll_to_gate_id = {}
    if isinstance(gates_df, pd.DataFrame):
        for _, row in gates_df.iterrows():
            ll = (float(row['lon']), float(row['lat']))
            gid = int(row['g_id']) if pd.notna(row['g_id']) else row['g_id']
            gate_id_to_ll[gid] = ll
            ll_to_gate_id[ll] = gid

    # 3) Weld connectors
    conn_df = out.get('gateB_connectors')
    if isinstance(conn_df, pd.DataFrame):
        for _, row in conn_df.iterrows():
            g_ll = gate_id_to_ll.get(row['g_id'])
            s_raw = row['sea_node_id']
            try:
                if isinstance(s_raw, str):
                    s_ll = tuple(map(float, s_raw.split(',')))  # (lon,lat)
                else:
                    s_ll = s_raw

                if g_ll and s_ll:
                    dist = np.sqrt((g_ll[0]-s_ll[0])**2 + (g_ll[1]-s_ll[1])**2)
                    G.add_edge(g_ll, s_ll, weight=dist, type='connector')
                    G.nodes[g_ll]['type'] = 'gate'
            except Exception:
                continue

    valid_gates = [ll for ll in gate_id_to_ll.values() if ll in G and G.degree(ll) > 0]
    print(f" 最終結果：節點={G.number_of_nodes()}, 邊={G.number_of_edges()}")
    print(f" 成功焊接進圖中的 Gate 數量: {len(valid_gates)}")

    if len(valid_gates) < 2:
        print(" 焊接失敗/可用 Gate 太少，請確認 Gate 與 Sea Node 的座標格式是否一致。")
        return None

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(valid_gates), size=2, replace=False)
    start_node = valid_gates[int(idx[0])]
    end_node   = valid_gates[int(idx[1])]

    print(f" 選到 Gate pair: {ll_to_gate_id.get(start_node)} -> {ll_to_gate_id.get(end_node)}")

    try:
        path = nx.astar_path(
            G, start_node, end_node,
            heuristic=lambda n1, n2: np.sqrt((n1[0]-n2[0])**2 + (n1[1]-n2[1])**2),
            weight='weight'
        )
        edges_in_path = list(zip(path, path[1:]))
        conn_used = sum(1 for u, v in edges_in_path if G[u][v].get('type') == 'connector')
        print(f" 導航成功！路徑點數: {len(path)}, 跨越 Connector 次數: {conn_used}")
        return path
    except Exception as e:
        print(f" 導航失敗: {e}")
        return None

# 用法：每次都會換（seed 不給就真隨機）
path = test_gate_navigation_random_pair(out, seed=None)


 最終結果：節點=1967, 邊=3168
 成功焊接進圖中的 Gate 數量: 24
 選到 Gate pair: 248 -> 245
 導航成功！路徑點數: 5, 跨越 Connector 次數: 2


#### Visualization with webbrowser

In [ ]:
import folium
from folium import plugins
import webbrowser
import os
import pandas as pd

def plot_path_folium_enhanced_v3(out, path, filename="nav_full_network_map.html"):
    # 1. 地圖初始化
    bbox = out.get('bbox_ll', (100, 10, 135, 50))
    center_lat = (bbox[1] + bbox[3]) / 2
    center_lon = (bbox[0] + bbox[2]) / 2
    
    m = folium.Map(
        location=[center_lat, center_lon], 
        zoom_start=6, 
        tiles='https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png',
        attr='&copy; OpenStreetMap contributors'
    )

    # --- 建立座標對照表 ---
    # 建立 C_nodes 的對照表 (用於 C_edges)
    c_nodes_map = out.get('C_nodes', {})
    # 建立 S_nodes 的對照表 (用於 S_edges)
    s_nodes_df = out.get('S_nodes')
    s_nodes_map = {}
    if isinstance(s_nodes_df, pd.DataFrame):
        for _, row in s_nodes_df.iterrows():
            s_nodes_map[row['node_id']] = (row['lon'], row['lat'])

    def get_pos(node, ref_map):
        # 如果 node 本身就是座標 (tuple), 直接回傳
        if isinstance(node, (tuple, list)) and len(node) >= 2:
            return node
        # 否則去對照表查
        return ref_map.get(node)

    
    # 2. 繪製 C-C Edges (Coastal)  —— 修正版
    cc_layer = folium.FeatureGroup(name="C-C Edges (Coastal)", show=True)
    # 2. 繪製 C-C Edges (Coastal) —— 正確版（支援 C_edges DataFrame）
    cc_layer = folium.FeatureGroup(name="C-C Edges (Coastal)", show=True)

    c_nodes_df = out.get("C_nodes", None)
    c_edges_df = out.get("C_edges", None)

    if isinstance(c_nodes_df, pd.DataFrame) and isinstance(c_edges_df, pd.DataFrame):
        # c_id -> (lon,lat)
        c_map = {
            int(r["c_id"]): (float(r["lon"]), float(r["lat"]))
            for _, r in c_nodes_df.iterrows()
        }

        drawn = 0
        for _, e in c_edges_df.iterrows():
            u = int(e["u"])
            v = int(e["v"])
            p1 = c_map.get(u)
            p2 = c_map.get(v)
            if p1 is None or p2 is None:
                continue

            folium.PolyLine(
                locations=[[p1[1], p1[0]], [p2[1], p2[0]]],
                weight=2,
                opacity=0.6,
                dash_array="5,5",
            ).add_to(cc_layer)
            drawn += 1

        print(f"[viz] C-C edges drawn: {drawn}/{len(c_edges_df)}")
    else:
        print("[viz] C_nodes 或 C_edges 不是 DataFrame，跳過 C-C edges 繪製")

    cc_layer.add_to(m)


    


    # 3. 繪製 Sea Edges (S_edges)
    sea_layer = folium.FeatureGroup(name='Sea Edges (Network)', show=True)
    sea_edges = out.get('S_edges', [])
    for edge in sea_edges:
        if len(edge) >= 2:
            u_pos = get_pos(edge[0], s_nodes_map)
            v_pos = get_pos(edge[1], s_nodes_map)
            if u_pos and v_pos:
                folium.PolyLine(
                    locations=[[u_pos[1], u_pos[0]], [v_pos[1], v_pos[0]]], 
                    color="#337aff", weight=1.5, opacity=0.5
                ).add_to(sea_layer)
    sea_layer.add_to(m)

    # 4. 繪製 Sea Nodes
    nodes_layer = folium.FeatureGroup(name='Sea Nodes (Grid)', show=False)
    if isinstance(s_nodes_df, pd.DataFrame):
        for _, row in s_nodes_df.iterrows():
            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=2, color='deepskyblue', fill=True, fill_opacity=0.6,
                popup=f"Node ID: {row.get('node_id', 'N/A')}"
            ).add_to(nodes_layer)
    nodes_layer.add_to(m)

    # 5. 繪製 Gate B
    gate_layer = folium.FeatureGroup(name='Gate B (Entry/Exit)', show=True)
    gates_df = out.get('Gate_B_kept_gates')
    if isinstance(gates_df, pd.DataFrame):
        for _, row in gates_df.iterrows():
            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=5, color='purple', fill=True, fill_opacity=0.8,
                popup=f"Gate UID: {row.get('gate_uid', 'N/A')}"
            ).add_to(gate_layer)
    gate_layer.add_to(m)

    # 6. 繪製 A* 導航路徑
    if path:
        path_latlon = [[p[1], p[0]] for p in path]
        folium.PolyLine(locations=path_latlon, color='red', weight=6, opacity=0.9).add_to(m)
        folium.Marker(location=path_latlon[0], icon=folium.Icon(color='green', icon='ship', prefix='fa')).add_to(m)
        folium.Marker(location=path_latlon[-1], icon=folium.Icon(color='orange', icon='anchor', prefix='fa')).add_to(m)

    # 7. 功能增強
    folium.LayerControl(collapsed=False).add_to(m)
    plugins.Fullscreen().add_to(m)
    plugins.MeasureControl(position='topleft', primary_length_unit='nauticalmiles').add_to(m)
    plugins.MiniMap(toggle_display=True).add_to(m)

    # 8. 存檔與開啟
    m.save(filename)
    print(f"地圖已生成：{os.path.realpath(filename)}")
    webbrowser.open("file://" + os.path.realpath(filename))

# 執行繪圖
plot_path_folium_enhanced_v3(out, path)

[viz] C-C edges drawn: 7587/7587
地圖已生成：C:\Users\slab\Desktop\Slab Project\Stage2 ETA\nav_full_network_map.html


In [84]:
print(type(out.get("C_nodes")), getattr(out.get("C_nodes"), "columns", None))
print("C_edges sample:", out.get("C_edges", [])[:5])
print("bbox_ll:", out.get("bbox_ll"))


<class 'pandas.core.frame.DataFrame'> Index(['c_id', 'ring_id', 'lon', 'lat', 'x_m', 'y_m', 's_km'], dtype='object')
C_edges sample:    ring_id  u  v  length_km
0        0  0  1  19.998047
1        0  1  2  20.000000
2        0  2  3  20.000000
3        0  3  4  20.000000
4        0  4  5  20.000000
bbox_ll: (100.0, 10, 135.0, 50)


#### Try random gateb to c node

In [31]:
import folium, webbrowser
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter
from math import radians, sin, cos, asin, sqrt

from routing_map.c_gateb_connectors import (
    build_cnode_gateb_connectors_nearest,
    add_cnode_gateb_connectors_to_graph,
)

# ---------------------------
# helpers
# ---------------------------
def in_bbox(p, bbox_ll):
    if bbox_ll is None:
        return True
    min_lon, min_lat, max_lon, max_lat = map(float, bbox_ll)
    lon, lat = float(p[0]), float(p[1])
    return (min_lon <= lon <= max_lon) and (min_lat <= lat <= max_lat)

def safe_df(out, name):
    df = out.get(name, None)
    if df is None:
        return None
    try:
        return df if len(df) > 0 else None
    except Exception:
        return None

def build_gate_xy(out):
    gate_df = safe_df(out, "Gate_all_cov")
    if gate_df is None:
        gate_df = safe_df(out, "Gate_all")
    if gate_df is None or "g_id" not in gate_df.columns:
        return {}
    return {int(r["g_id"]): (float(r["lon"]), float(r["lat"])) for _, r in gate_df.iterrows()}

def get_gateB_df(out, gate_xy):
    gb_obj = out.get("Gate_B_kept_gates", None)
    dfGB = None

    if isinstance(gb_obj, pd.DataFrame):
        dfGB = gb_obj.copy()
        if "lon" not in dfGB.columns or "lat" not in dfGB.columns:
            if "g_id" in dfGB.columns and gate_xy:
                dfGB["lon"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[0])
                dfGB["lat"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[1])
        dfGB = dfGB.dropna(subset=["lon", "lat"])

    elif isinstance(gb_obj, (list, set, tuple, np.ndarray)):
        gids = [int(x) for x in gb_obj]
        rows = []
        for gid in gids:
            if gid in gate_xy:
                lon, lat = gate_xy[gid]
                rows.append({"g_id": gid, "lon": lon, "lat": lat})
        dfGB = pd.DataFrame(rows)

    if dfGB is None:
        gb2 = safe_df(out, "Gate_B")
        if gb2 is not None:
            dfGB = gb2.copy()

    return dfGB if (dfGB is not None and len(dfGB) > 0) else None

EARTH_R_KM = 6371.0088

from math import radians, sin, cos, asin, sqrt

# 處理換日線問題 helper
def _wrap_dlon_deg(lon2: float, lon1: float) -> float:
    """
    Return the shortest signed longitude difference (degrees) in [-180, 180].
    This makes distance calculations dateline-safe.
    """
    return (float(lon2) - float(lon1) + 180.0) % 360.0 - 180.0

def haversine_km(p1, p2) -> float:
    """
    Haversine distance in kilometers between two lon/lat points.
    Dateline-safe: uses wrapped longitude difference (shortest angle).
    p1, p2: (lon, lat)
    """
    lon1, lat1 = map(float, p1)
    lon2, lat2 = map(float, p2)

    dlon = radians(_wrap_dlon_deg(lon2, lon1))
    dlat = radians(lat2 - lat1)

    lat1r = radians(lat1)
    lat2r = radians(lat2)

    a = sin(dlat / 2.0) ** 2 + cos(lat1r) * cos(lat2r) * sin(dlon / 2.0) ** 2
    # guard tiny numerical drift (optional but harmless)
    a = min(1.0, max(0.0, a))

    return 2.0 * EARTH_R_KM * asin(sqrt(a))


def parse_node_id_str(s):
    try:
        a, b = s.split(",")
        return float(a), float(b)
    except Exception:
        return None

def edge_to_lonlat(e, *, nodes_df=None, idx_to_lonlat_fn=None):
    """
    Return ((lon1,lat1),(lon2,lat2)) or None.
    Supports:
      1) (u_idx, v_idx) using nodes_df + idx_to_lonlat_fn
      2) ((lon,lat),(lon,lat))
      3) ("lon,lat","lon,lat")
    """
    if not isinstance(e, (list, tuple)) or len(e) < 2:
        return None
    a, b = e[0], e[1]

    if isinstance(a, (int, np.integer)) and isinstance(b, (int, np.integer)):
        if nodes_df is None or idx_to_lonlat_fn is None:
            return None
        try:
            return (idx_to_lonlat_fn(a), idx_to_lonlat_fn(b))
        except Exception:
            return None

    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)) and len(a) >= 2 and len(b) >= 2:
        try:
            return ((float(a[0]), float(a[1])), (float(b[0]), float(b[1])))
        except Exception:
            return None

    if isinstance(a, str) and isinstance(b, str):
        p1 = parse_node_id_str(a)
        p2 = parse_node_id_str(b)
        if p1 and p2:
            return (p1, p2)

    return None

def pick_random_pair_with_path(G, A, B, rng, max_tries=30):
    """
    A, B: list of node tuples (lon,lat) already filtered in AOI & present in G
    Try random pairs until a path is found.
    """
    if len(A) == 0 or len(B) == 0:
        return None, None, None

    for _ in range(int(max_tries)):
        a = A[int(rng.integers(len(A)))]
        b = B[int(rng.integers(len(B)))]
        try:
            path = nx.astar_path(G, a, b, heuristic=lambda n1, n2: haversine_km(n1, n2), weight="weight")
            return a, b, path
        except Exception:
            continue
    return None, None, None

# ---------------------------
# main: open debug map
# ---------------------------
def open_routing_debug_map_clean_notebook(
    out,
    *,
    html_path="aoi_debug_map.html",
    zoom_start=5,
    seed=None,
    # viz caps
    c_sample=8000,
    s_sample=None,
    max_sea_edges_viz=6000,
    # graph caps
    max_sea_edges_graph=None,
    max_cc_edges_graph=None,
    # C<->GateB bridge
    c_to_gateB_max_deg_dist=None,  # None 建議先不設；想避免超長橋再設 0.3~0.6
    # route direction
    route_mode="gateB_to_coastal",  # "gateB_to_coastal" or "coastal_to_gateB"
):
    bbox_ll = out.get("bbox_ll", None)
    if bbox_ll is None:
        raise ValueError("out['bbox_ll'] is required.")
    min_lon, min_lat, max_lon, max_lat = map(float, bbox_ll)
    center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]
    rng = np.random.default_rng(seed)

    # --- data ---
    C_nodes = safe_df(out, "C_nodes")
    C_edges_df = safe_df(out, "C_edges")   # confirmed DF ring_id,u,v,length_km
    S_nodes = safe_df(out, "S_nodes")
    S_edges = out.get("S_edges", None)
    gate_xy = build_gate_xy(out)
    dfGB = get_gateB_df(out, gate_xy)
    dfGB_conn = safe_df(out, "gateB_connectors")

    # --- map ---
    m = folium.Map(location=center, zoom_start=zoom_start, control_scale=True)
    folium.Rectangle(bounds=[[min_lat, min_lon], [max_lat, max_lon]], fill=False, weight=3, opacity=0.9).add_to(m)
    m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    def circle_layer(df, name, radius, show=True):
        fg = folium.FeatureGroup(name=name, show=show)
        for _, r in df.iterrows():
            p = (float(r["lon"]), float(r["lat"]))
            if not in_bbox(p, bbox_ll):
                continue
            folium.CircleMarker([p[1], p[0]], radius=radius).add_to(fg)
        fg.add_to(m)

    # --- node layers ---
    if isinstance(C_nodes, pd.DataFrame) and len(C_nodes) > 0:
        nC = len(C_nodes)
        dfC_plot = C_nodes.sample(min(int(c_sample), nC), random_state=7) if nC > c_sample else C_nodes
        circle_layer(dfC_plot, f"C_nodes (sample {len(dfC_plot)}/{nC})", radius=1, show=True)

    if isinstance(dfGB, pd.DataFrame) and len(dfGB) > 0:
        circle_layer(dfGB, f"Gate_B ({len(dfGB)})", radius=6, show=True)

    if isinstance(S_nodes, pd.DataFrame) and len(S_nodes) > 0:
        nS = len(S_nodes)
        if s_sample is not None and nS > int(s_sample):
            dfS_plot = S_nodes.sample(int(s_sample), random_state=7)
            title = f"S_nodes (sample {len(dfS_plot)}/{nS})"
        else:
            dfS_plot = S_nodes
            title = f"S_nodes ({nS})"
        circle_layer(dfS_plot, title, radius=3, show=True)
    
    

    # --- sea edges layer (viz) ---
    def sea_lonlat_by_idx(i):
        s = S_nodes.iloc[int(i)]
        return (float(s["lon"]), float(s["lat"]))

    if isinstance(S_nodes, pd.DataFrame) and S_edges is not None and len(S_edges) > 0:
        fgE = folium.FeatureGroup(name=f"S_edges (show {min(len(S_edges), max_sea_edges_viz)}/{len(S_edges)})", show=True)
        take = S_edges[:max_sea_edges_viz] if len(S_edges) > max_sea_edges_viz else S_edges
        drawn = 0
        for e in take:
            seg = edge_to_lonlat(e, nodes_df=S_nodes, idx_to_lonlat_fn=sea_lonlat_by_idx)
            if seg is None:
                continue
            u, v = seg
            if not (in_bbox(u, bbox_ll) or in_bbox(v, bbox_ll)):
                continue
            folium.PolyLine([[u[1], u[0]], [v[1], v[0]]], color="#3352ff", weight=2, opacity=0.8).add_to(fgE)
            drawn += 1
        fgE.add_to(m)
        print(f"[viz] sea edges drawn: {drawn}/{len(take)}")

    # --- C-C edges layer (viz) ---
    if isinstance(C_nodes, pd.DataFrame) and isinstance(C_edges_df, pd.DataFrame):
        cc_layer = folium.FeatureGroup(name="C-C edges (Coastal)", show=False)
        c_map = {int(r["c_id"]): (float(r["lon"]), float(r["lat"])) for _, r in C_nodes.iterrows()}

        drawn = 0
        for _, e in C_edges_df.iterrows():
            p1 = c_map.get(int(e["u"]))
            p2 = c_map.get(int(e["v"]))
            if p1 is None or p2 is None:
                continue
            if not (in_bbox(p1, bbox_ll) or in_bbox(p2, bbox_ll)):
                continue
            folium.PolyLine([[p1[1], p1[0]], [p2[1], p2[0]]], weight=2, opacity=0.6, dash_array="5,5").add_to(cc_layer)
            drawn += 1

        cc_layer.add_to(m)
        print(f"[viz] C-C edges drawn: {drawn}/{len(C_edges_df)}")

    # --- GateB -> Sea connectors layer (viz) ---
    if isinstance(dfGB_conn, pd.DataFrame) and isinstance(S_nodes, pd.DataFrame) and gate_xy:
        fgConn = folium.FeatureGroup(name=f"GateB→Sea connectors ({len(dfGB_conn)})", show=True)
        drawn = 0
        for _, r in dfGB_conn.iterrows():
            try:
                gid = int(r["g_id"])
                if gid not in gate_xy:
                    continue
                gb = gate_xy[gid]
                if not in_bbox(gb, bbox_ll):
                    continue
                sea = sea_lonlat_by_idx(int(r["sea_idx"]))
                if not (in_bbox(gb, bbox_ll) or in_bbox(sea, bbox_ll)):
                    continue
                folium.PolyLine([[gb[1], gb[0]], [sea[1], sea[0]]],color="green", weight=2, opacity=0.95).add_to(fgConn)
                drawn += 1
            except Exception:
                continue
        fgConn.add_to(m)
        print(f"[viz] GateB→Sea connectors drawn: {drawn}/{len(dfGB_conn)}")
        

    # =========================
    # Build C<->GateB bridge connectors (module)
    # =========================
    cgb_df = None
    if isinstance(C_nodes, pd.DataFrame) and isinstance(dfGB, pd.DataFrame):
        cgb_df = build_cnode_gateb_connectors_nearest(
            C_nodes,
            dfGB[["g_id", "lon", "lat"]].rename(columns={"lon": "lon", "lat": "lat"}),
            bbox_ll=bbox_ll,
            max_deg_dist=c_to_gateB_max_deg_dist,
        )
        # draw layer
        fgCGB = folium.FeatureGroup(name=f"C↔GateB connectors (nearest) ({len(cgb_df)})", show=True)
        for _, r in cgb_df.iterrows():
            c = (float(r["c_lon"]), float(r["c_lat"]))
            gb = (float(r["g_lon"]), float(r["g_lat"]))
            folium.PolyLine([[c[1], c[0]], [gb[1], gb[0]]], weight=2, opacity=0.9).add_to(fgCGB)
        fgCGB.add_to(m)
        print(f"[viz] C↔GateB connectors built: {len(cgb_df)}")

    # =========================
    # Build ONE routing graph
    # =========================
    G = nx.Graph()

    # 1) sea edges
    if isinstance(S_nodes, pd.DataFrame) and S_edges is not None and len(S_edges) > 0:
        take = S_edges
        if max_sea_edges_graph is not None and len(take) > int(max_sea_edges_graph):
            take = take[:int(max_sea_edges_graph)]

        added = 0
        for e in take:
            seg = edge_to_lonlat(e, nodes_df=S_nodes, idx_to_lonlat_fn=sea_lonlat_by_idx)
            if seg is None:
                continue
            u, v = seg
            if not (in_bbox(u, bbox_ll) or in_bbox(v, bbox_ll)):
                continue
            G.add_edge(u, v, weight=haversine_km(u, v), etype="sea")
            added += 1
        print(f"[route] graph sea edges added: {added}/{len(take)}")

    # 2) coastal cc edges
    if isinstance(C_nodes, pd.DataFrame) and isinstance(C_edges_df, pd.DataFrame):
        c_map = {int(r["c_id"]): (float(r["lon"]), float(r["lat"])) for _, r in C_nodes.iterrows()}

        cc_df = C_edges_df
        if max_cc_edges_graph is not None and len(cc_df) > int(max_cc_edges_graph):
            cc_df = cc_df.iloc[:int(max_cc_edges_graph)]

        added = 0
        for _, e in cc_df.iterrows():
            p1 = c_map.get(int(e["u"]))
            p2 = c_map.get(int(e["v"]))
            if p1 is None or p2 is None:
                continue
            if not (in_bbox(p1, bbox_ll) or in_bbox(p2, bbox_ll)):
                continue
            len_km = float(e["length_km"]) if "length_km" in e and pd.notna(e["length_km"]) else haversine_km(p1, p2)
            G.add_edge(p1, p2, weight=len_km, etype="cc")
            added += 1
        print(f"[route] graph cc edges added: {added}/{len(cc_df)}")

    # 3) GateB -> Sea connectors
    if isinstance(dfGB_conn, pd.DataFrame) and isinstance(S_nodes, pd.DataFrame) and gate_xy:
        added = 0
        for _, r in dfGB_conn.iterrows():
            try:
                gid = int(r["g_id"])
                if gid not in gate_xy:
                    continue
                gb = gate_xy[gid]
                if not in_bbox(gb, bbox_ll):
                    continue
                sea = sea_lonlat_by_idx(int(r["sea_idx"]))
                if not (in_bbox(gb, bbox_ll) or in_bbox(sea, bbox_ll)):
                    continue
                G.add_edge(gb, sea, weight= haversine_km(gb, sea), etype="gb_sea")
                added += 1
            except Exception:
                continue
        print(f"[route] graph GateB→Sea edges added: {added}/{len(dfGB_conn)}")

    # 4) C <-> GateB bridge connectors (NEW)
    if isinstance(cgb_df, pd.DataFrame) and len(cgb_df) > 0:
        # 在 add_cnode_gateb_connectors_to_graph 前面加
        cgb_df["dist_km"] = cgb_df.apply(
            lambda r: haversine_km((r["c_lon"], r["c_lat"]), (r["g_lon"], r["g_lat"])),
            axis=1
        )
        added = add_cnode_gateb_connectors_to_graph(G, cgb_df, etype="c_gb", weight_col="dist_km")
        print(f"[route] graph C↔GateB bridge edges added: {added}")

    # =========================
    # Pick random endpoints INSIDE AOI and route
    # =========================
    coastal_candidates = []
    if isinstance(C_nodes, pd.DataFrame):
        for _, r in C_nodes.iterrows():
            p = (float(r["lon"]), float(r["lat"]))
            if in_bbox(p, bbox_ll) and (p in G) and (G.degree(p) > 0):
                coastal_candidates.append(p)

    gateb_candidates = []
    if isinstance(dfGB, pd.DataFrame):
        for _, r in dfGB.iterrows():
            p = (float(r["lon"]), float(r["lat"]))
            if in_bbox(p, bbox_ll) and (p in G) and (G.degree(p) > 0):
                gateb_candidates.append(p)

    if len(coastal_candidates) == 0 or len(gateb_candidates) == 0:
        print(f"[route] skip: coastal_candidates={len(coastal_candidates)}, gateb_candidates={len(gateb_candidates)}")
    else:
        if route_mode == "coastal_to_gateB":
            A, B = coastal_candidates, gateb_candidates
            start_label, end_label = "START (Coastal)", "END (GateB)"
            layer_name = "Route: random Coastal → random GateB"
        elif route_mode == "gateB_to_coastal":
            A, B = gateb_candidates, coastal_candidates
            start_label, end_label = "START (GateB)", "END (Coastal)"
            layer_name = "Route: random GateB → random Coastal"
        elif route_mode == "gateB_to_gateB":
            A, B = gateb_candidates, gateb_candidates
            start_label, end_label = "START (GateB)", "END (GateB)"
            layer_name = "Route: random GateB → random GateB"
        else:
            A, B = coastal_candidates, coastal_candidates
            start_label, end_label = "START (Coastal)", "END (Coastal)"
            layer_name = "Route: random Coastal → random Coastal"
    

        start, end, path = pick_random_pair_with_path(G, A, B, rng, max_tries=30)

        if path is None:
            print("[route] FAIL: no path found after tries (graph may be disconnected or endpoints unlucky)")
        else:
            etypes = [G[u][v].get("etype") for u, v in zip(path, path[1:])]
            cnt = Counter(etypes)
            print(f"[route] OK | points={len(path)} | etypes={dict(cnt)}")

            fgR = folium.FeatureGroup(name=layer_name, show=True)
            folium.Marker([start[1], start[0]], tooltip=start_label).add_to(fgR)
            folium.Marker([end[1], end[0]], tooltip=end_label).add_to(fgR)
            folium.PolyLine([[p[1], p[0]] for p in path], color="#ff3333", weight=6, opacity=0.95).add_to(fgR)
            fgR.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    html_path = Path(html_path).resolve()
    m.save(str(html_path))
    webbrowser.open(html_path.as_uri())
    return html_path


# === 直接呼叫 ===
open_routing_debug_map_clean_notebook(
    out,
    html_path="aoi_debug_map.html",
    zoom_start=5,
    seed=None,  # None = 真隨機；給整數可重現
    route_mode="gateB_to_coastal",  # 或 "coastal_to_gateB"
    c_to_gateB_max_deg_dist=None,   # 想避免超長橋再設 0.3~0.6
)


[viz] sea edges drawn: 1478/1536
[viz] C-C edges drawn: 6482/13095
[viz] GateB→Sea connectors drawn: 157/157
[viz] C↔GateB connectors built: 77
[route] graph sea edges added: 1478/1536
[route] graph cc edges added: 6482/13095
[route] graph GateB→Sea edges added: 157/157
[route] graph C↔GateB bridge edges added: 77
[route] OK | points=56 | etypes={'gb_sea': 2, 'sea': 19, 'cc': 34}


WindowsPath('C:/Users/slab/Desktop/Slab Project/Stage2 ETA/aoi_debug_map.html')

### P2P routing

#### Add snap.py, routing_graph.py, simplifier, repairer

In [13]:
import folium, webbrowser
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
from collections import Counter

from shapely.geometry import LineString, box
from shapely.ops import transform as shp_transform

# === modules ===
from routing_map.path_simplifier import simplify_path_visibility
from routing_map.routing_graph import build_base_graph, haversine_km
from routing_map.c_gateb_connectors import (
    build_cnode_gateb_connectors_nearest,
    add_cnode_gateb_connectors_to_graph,
)
from routing_map.snap import snap_pair_component_aware, inject_point_edges
from routing_map.repairer import PathRepairer, RepairConfig

#  snap-link repair helper (you said it's already ready)
from routing_map.snap_link_repair import repair_snap_link_ll_if_needed

from routing_map.metrics import path_length_km_nm, format_distance
from routing_map.snap_link_repair import repair_snap_link_ll_if_needed



# ---------------------------
# helpers
# ---------------------------
def in_bbox(p, bbox_ll):
    if bbox_ll is None:
        return True
    min_lon, min_lat, max_lon, max_lat = map(float, bbox_ll)
    lon, lat = float(p[0]), float(p[1])
    return (min_lon <= lon <= max_lon) and (min_lat <= lat <= max_lat)

def safe_df(out, name):
    df = out.get(name, None)
    if df is None:
        return None
    try:
        return df if len(df) > 0 else None
    except Exception:
        return None

def build_gate_xy(out):
    gate_df = safe_df(out, "Gate_all_cov")
    if gate_df is None:
        gate_df = safe_df(out, "Gate_all")
    if gate_df is None or "g_id" not in gate_df.columns:
        return {}
    return {int(r["g_id"]): (float(r["lon"]), float(r["lat"])) for _, r in gate_df.iterrows()}

def get_gateB_df(out, gate_xy):
    gb_obj = out.get("Gate_B_kept_gates", None)
    dfGB = None

    if isinstance(gb_obj, pd.DataFrame):
        dfGB = gb_obj.copy()
        if "lon" not in dfGB.columns or "lat" not in dfGB.columns:
            if "g_id" in dfGB.columns and gate_xy:
                dfGB["lon"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[0])
                dfGB["lat"] = dfGB["g_id"].map(lambda x: gate_xy.get(int(x), (np.nan, np.nan))[1])
        dfGB = dfGB.dropna(subset=["lon", "lat"])

    elif isinstance(gb_obj, (list, set, tuple, np.ndarray)):
        gids = [int(x) for x in gb_obj]
        rows = []
        for gid in gids:
            if gid in gate_xy:
                lon, lat = gate_xy[gid]
                rows.append({"g_id": gid, "lon": lon, "lat": lat})
        dfGB = pd.DataFrame(rows)

    if dfGB is None:
        gb2 = safe_df(out, "Gate_B")
        if gb2 is not None:
            dfGB = gb2.copy()

    return dfGB if (dfGB is not None and len(dfGB) > 0) else None

def parse_node_id_str(s):
    try:
        a, b = s.split(",")
        return float(a), float(b)
    except Exception:
        return None

def edge_to_lonlat(e, *, nodes_df=None, idx_to_lonlat_fn=None):
    if not isinstance(e, (list, tuple)) or len(e) < 2:
        return None
    a, b = e[0], e[1]

    if isinstance(a, (int, np.integer)) and isinstance(b, (int, np.integer)):
        if nodes_df is None or idx_to_lonlat_fn is None:
            return None
        try:
            return (idx_to_lonlat_fn(a), idx_to_lonlat_fn(b))
        except Exception:
            return None

    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)) and len(a) >= 2 and len(b) >= 2:
        try:
            return ((float(a[0]), float(a[1])), (float(b[0]), float(b[1])))
        except Exception:
            return None

    if isinstance(a, str) and isinstance(b, str):
        p1 = parse_node_id_str(a)
        p2 = parse_node_id_str(b)
        if p1 and p2:
            return (p1, p2)

    return None

def unwrap_lon(lon, ref_lon):
    lon = float(lon); ref_lon = float(ref_lon)
    d = lon - ref_lon
    if d > 180: lon -= 360
    if d < -180: lon += 360
    return lon

def route_polyline_dateline_safe(path_ll):
    if not path_ll:
        return []
    out = []
    ref = float(path_ll[0][0])
    for lon, lat in path_ll:
        lon_u = unwrap_lon(lon, ref)
        out.append((lon_u, float(lat)))
        ref = lon_u
    return out


# --- projection utilities (USE out["proj"] to match layers CRS) ---
def _make_ll_m_projectors_from_out(out):
    """
    Return:
      ll2m_xy(lon,lat)->(x,y)      # for repairer (positional lon,lat)
      m2ll_xy(x,y)->(lon,lat)
      ll2m_tuple((lon,lat))->(x,y) # for simplifier (tuple)
      m2ll_tuple((x,y))->(lon,lat)
    """
    proj = out.get("proj", None)
    if proj is None:
        raise ValueError("out['proj'] not found. build_aoi() should return 'proj'.")

    def _apply(fn, a, b):
        if hasattr(fn, "transform") and callable(getattr(fn, "transform")):
            return fn.transform(a, b)
        if callable(fn):
            try:
                return fn(a, b)
            except TypeError:
                return fn((a, b))
        raise TypeError(f"Projector {type(fn)} not callable and no .transform")

    candidates = [
        ("ll_to_xy", "xy_to_ll"),
        ("ll_to_m", "m_to_ll"),
        ("to_m", "to_ll"),
        ("fwd", "inv"),
        ("forward", "inverse"),
    ]

    for a, b in candidates:
        if hasattr(proj, a) and hasattr(proj, b):
            f = getattr(proj, a)
            g = getattr(proj, b)

            def ll2m_xy(lon, lat, _f=f):
                x, y = _apply(_f, float(lon), float(lat))
                return (float(x), float(y))

            def m2ll_xy(x, y, _g=g):
                lon, lat = _apply(_g, float(x), float(y))
                return (float(lon), float(lat))

            ll2m_tuple = lambda p: ll2m_xy(p[0], p[1])
            m2ll_tuple = lambda q: m2ll_xy(q[0], q[1])
            return ll2m_xy, m2ll_xy, ll2m_tuple, m2ll_tuple

    if hasattr(proj, "transform") and callable(getattr(proj, "transform")):
        raise ValueError("proj.transform exists but inverse isn't inferable; please provide proj with inverse method.")

    raise ValueError("Cannot infer projection methods from out['proj'].")


def _get_collision_metric(out):
    layers = out.get("layers", None)
    if isinstance(layers, dict) and layers.get("COLLISION_M") is not None:
        return layers["COLLISION_M"]

    c = out.get("COLLISION_M", None)
    if c is None:
        c = out.get("collision_m", None)
    if c is not None:
        return c

    cp = out.get("collision_prep", None)
    if cp is not None and hasattr(cp, "context") and cp.context is not None:
        return cp.context

    return out.get("collision", None)


from shapely.geometry import box
import numpy as np
from shapely.geometry import box

def _get_densified_metric_box(bbox_ll, ll2m_xy, step_deg=2.0, pad_m=50_000.0):
    """
    沿著 AOI 的四個邊界進行密集取樣，然後找出 Metric 空間中真正的 min/max xy。
    解決大範圍投影造成的「弧線切除」問題。
    """
    min_lon, min_lat, max_lon, max_lat = map(float, bbox_ll)
    
    xs, ys = [], []
    
    # 建立經度與緯度的取樣點
    lons = np.arange(min_lon, max_lon + step_deg, step_deg)
    if lons[-1] != max_lon: lons = np.append(lons, max_lon)
    
    lats = np.arange(min_lat, max_lat + step_deg, step_deg)
    if lats[-1] != max_lat: lats = np.append(lats, max_lat)
    
    # 定義要檢查的邊界點 (上緣、下緣、左緣、右緣)
    # 1. Top & Bottom edges (沿著經度走)
    for lon in lons:
        # Bottom edge
        x, y = ll2m_xy(lon, min_lat)
        xs.append(x); ys.append(y)
        # Top edge
        x, y = ll2m_xy(lon, max_lat)
        xs.append(x); ys.append(y)
        
    # 2. Left & Right edges (沿著緯度走)
    for lat in lats:
        # Left edge
        x, y = ll2m_xy(min_lon, lat)
        xs.append(x); ys.append(y)
        # Right edge
        x, y = ll2m_xy(max_lon, lat)
        xs.append(x); ys.append(y)
        
    # 找出真正的 Metric 邊界
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    
    # 加上 Padding 並回傳 Metric Box
    return box(min_x, min_y, max_x, max_y).buffer(pad_m)

def _clip_collision_to_aoi_bbox(collision_m, bbox_ll, ll2m_xy, pad_m=80_000.0):
    """
    Clip metric collision to AOI bbox (also in metric), robust to nonlinear/local projections:
    - Project ALL 4 bbox corners and take min/max in metric space.
    - Then build a metric box (+ optional pad) and intersect.

    NOTE:
      bbox_ll is (min_lon, min_lat, max_lon, max_lat) in lon/lat degrees.
    """
    if collision_m is None or bbox_ll is None:
        return collision_m

    min_lon, min_lat, max_lon, max_lat = map(float, bbox_ll)

    # If bbox crosses dateline in [-180,180] convention (rare in your current AOI),
    # you can either skip clip or handle split-box. For now, handle the simple case only.
    # (Your current bbox (10..150) doesn't cross, so this won't trigger.)
    if min_lon > max_lon:
        # conservative fallback: don't clip (avoid generating a wrong box)
        # You can implement split-box clipping later if you ever need dateline AOI.
        print("[collision] bbox crosses dateline; skip clip to avoid wrong AOI box")
        return collision_m

    # --- Project 4 corners (IMPORTANT) ---
    corners_ll = [
        (min_lon, min_lat),
        (min_lon, max_lat),
        (max_lon, min_lat),
        (max_lon, max_lat),
    ]

    xs, ys = [], []
    for lon, lat in corners_ll:
        try:
            x, y = ll2m_xy(lon, lat)
        except Exception as e:
            print("[collision] clip: ll2m failed on corner", (lon, lat), "-> keep original:", repr(e))
            return collision_m
        xs.append(float(x))
        ys.append(float(y))

    minx, maxx = min(xs), max(xs)
    miny, maxy = min(ys), max(ys)

    # Build AOI metric window (+ pad)
    #aoi_box_m = box(minx, miny, maxx, maxy).buffer(float(pad_m))
    aoi_box_m = _get_densified_metric_box(bbox_ll, ll2m_xy, step_deg=2.0, pad_m=pad_m)

    try:
        c2 = collision_m.intersection(aoi_box_m)
        if c2 is None or c2.is_empty:
            print("[collision] clip empty -> keep original")
            return collision_m
        return c2
    except Exception as e:
        print("[collision] clip failed -> keep original:", repr(e))
        return collision_m




def _geom_m_to_ll(geom_m, m2ll_xy):
    def _norm_lon(lon):
        lon = float(lon)
        # normalize to [-180, 180]
        while lon > 180.0:
            lon -= 360.0
        while lon < -180.0:
            lon += 360.0
        return lon

    def _xy_to_lonlat(x, y, z=None):
        a, b = m2ll_xy(float(x), float(y))

        a = float(a); b = float(b)

        # --- Auto-fix axis order if inverse returns (lat, lon) ---
        # Heuristic: lat must be [-90,90], lon must be [-180,180] (or 0..360 before norm)
        if abs(a) <= 90.0 and abs(b) <= 360.0 and abs(b) > 90.0:
            # looks like (lat, lon)
            lat, lon = a, b
        else:
            # assume (lon, lat)
            lon, lat = a, b

        lon = _norm_lon(lon)
        lat = max(-90.0, min(90.0, float(lat)))  # clamp safety
        return (lon, lat)

    return shp_transform(_xy_to_lonlat, geom_m)



# ---------------------------
# main
# ---------------------------
def open_routing_debug_map_p2p(
    out,
    origin_ll,
    dest_ll,
    *,
    html_path="aoi_p2p_map.html",
    zoom_start=5,

    c_sample=8000,
    s_sample=3000,
    max_sea_edges_viz=6000,

    include_sea=True,
    include_cc=True,
    include_gateb_sea=True,

    use_c_gateb_bridge=True,
    c_to_gateB_max_deg_dist=None,

    k_near=30,
    r_max_km_snap=150.0,
    k_inject=4,

    do_repair=True,
    do_simplify=True,
):
    bbox_ll = out.get("bbox_ll", None)
    if bbox_ll is None:
        raise ValueError("out['bbox_ll'] is required.")
    min_lon, min_lat, max_lon, max_lat = map(float, bbox_ll)
    center = [(min_lat + max_lat) / 2, (min_lon + max_lon) / 2]

    origin_ll = (float(origin_ll[0]), float(origin_ll[1]))
    dest_ll   = (float(dest_ll[0]), float(dest_ll[1]))

    C_nodes = safe_df(out, "C_nodes")
    S_nodes = safe_df(out, "S_nodes")
    S_edges = out.get("S_edges", None)
    C_edges_df = safe_df(out, "C_edges")

    gate_xy = build_gate_xy(out)
    dfGB = get_gateB_df(out, gate_xy)
    dfGB_conn = safe_df(out, "gateB_connectors")

    # --- build base graph ---
    G, stats = build_base_graph(
        out,
        include_sea=bool(include_sea),
        include_cc=bool(include_cc),
        include_gateb_sea=bool(include_gateb_sea),
        include_c_gateb=False,
        bbox_ll=bbox_ll,
        weight_unit="km",
    )
    print("[graph] base stats:", stats)

    # --- optional C↔GateB bridge ---
    cgb_df = None
    if use_c_gateb_bridge and isinstance(C_nodes, pd.DataFrame) and isinstance(dfGB, pd.DataFrame):
        cgb_df = build_cnode_gateb_connectors_nearest(
            C_nodes,
            dfGB[["g_id", "lon", "lat"]],
            bbox_ll=bbox_ll,
            max_deg_dist=c_to_gateB_max_deg_dist,
        )
        added = add_cnode_gateb_connectors_to_graph(G, cgb_df, etype="c_gb", weight_col="dist_deg")
        print(f"[graph] C↔GateB bridge edges added: {added}")

    # --- snap pair ---
    pair = snap_pair_component_aware(
        out,
        origin_ll, dest_ll,
        k_near=int(k_near),
        r_max_km=float(r_max_km_snap),
        k_inject=int(k_inject),
        prefer_ok_set=True,
        allow_fallback_non_ok=True,
        allow_radius_fallback=True,
        do_nudge=True,
    )

    start_key = getattr(pair.start, "p_used_ll", origin_ll)
    end_key   = getattr(pair.end,   "p_used_ll", dest_ll)
    start_key = (float(start_key[0]), float(start_key[1]))
    end_key   = (float(end_key[0]), float(end_key[1]))

    path = None
    path_ll_for_simplify = None
    path_simplified, simp_stats = None, None

    #  store inject-edge polylines (these are the "temporary point" links you care about)
    inject_start_ll = None   # path[0] -> path[1] (repaired if start_inject)
    inject_end_ll   = None   # path[-2] -> path[-1] (repaired if end_inject)

    if len(pair.start_pick) == 0 or len(pair.end_pick) == 0:
        print("[snap] FAIL:", pair.reason, pair.debug)
    else:
        inject_point_edges(G, start_key, pair.start_pick, k_inject=int(k_inject), etype="start_inject")
        inject_point_edges(G, end_key,   pair.end_pick,   k_inject=int(k_inject), etype="end_inject")
        print("[snap] OK:", pair.reason, pair.debug)
        print("[snap][start] local_entrance_aug:", pair.start.debug.get("local_entrance_aug"))
        print("[snap][end  ] local_entrance_aug:", pair.end.debug.get("local_entrance_aug"))

        # --- A* ---
        try:
            path = nx.astar_path(
                G,
                start_key,
                end_key,
                heuristic=lambda n1, n2: haversine_km(n1, n2),
                weight="weight",
            )
            etypes = [G[u][v].get("etype") for u, v in zip(path, path[1:])]
            print("[route] OK | points=", len(path), "| etypes=", dict(Counter(etypes)))
        except Exception as e:
            path = None
            print("[route] FAIL:", repr(e))

        # --- repair + simplify ---
        if path is None or len(path) < 2:
            print("[repair/simplify] skip: no valid path")
        else:
            ll2m_xy, m2ll_xy, ll2m_tuple, m2ll_tuple = _make_ll_m_projectors_from_out(out)

            def _ll_to_m_any(a, b=None):
                if b is None:
                    return ll2m_xy(float(a[0]), float(a[1]))
                return ll2m_xy(float(a), float(b))

            def _m_to_ll_any(a, b=None):
                if b is None:
                    return m2ll_xy(float(a[0]), float(a[1]))
                return m2ll_xy(float(a), float(b))

            collision = _get_collision_metric(out)
            if collision is None:
                print("[collision] not found -> skip repair/simplify")
                path_ll_for_simplify = [(float(p[0]), float(p[1])) for p in path]
            else:
                collision = _clip_collision_to_aoi_bbox(collision, bbox_ll, ll2m_xy, pad_m=80000.0)
                print("[collision] bounds:", getattr(collision, "bounds", None), "type:", getattr(collision, "geom_type", type(collision)))
                collision_used = collision

                path_ll = [(float(p[0]), float(p[1])) for p in path]

                if do_repair:
                    repairer_obj = PathRepairer(RepairConfig(
                        debug=True,
                        rb_n_samples=25,
                        rb_max_iter=60,
                        rb_push_step_m=250.0,
                        rb_smooth_lambda=0.35,
                    ))

                    # ----------------------------
                    # (1) Repair inject edges ONLY:
                    #     - start: path[0] -> path[1] if etype == "start_inject"
                    #     - end  : path[-2] -> path[-1] if etype == "end_inject"
                    # ----------------------------
                    u0, u1 = path[0], path[1]
                    v1, v0 = path[-2], path[-1]

                    et0 = G[u0][u1].get("etype") if G.has_edge(u0, u1) else None
                    et1 = G[v1][v0].get("etype") if G.has_edge(v1, v0) else None
                    print("[inject] first edge etype:", et0, "| last edge etype:", et1)

                    # start inject polyline
                    if et0 == "start_inject":
                        inject_start_ll = repair_snap_link_ll_if_needed(
                            (float(u0[0]), float(u0[1])),
                            (float(u1[0]), float(u1[1])),
                            collision_m=collision,
                            ll_to_m=_ll_to_m_any,
                            m_to_ll=_m_to_ll_any,
                            repairer_obj=repairer_obj,
                        )
                    else:
                        inject_start_ll = [(float(u0[0]), float(u0[1])), (float(u1[0]), float(u1[1]))]

                    # end inject polyline
                    if et1 == "end_inject":
                        inject_end_ll = repair_snap_link_ll_if_needed(
                            (float(v1[0]), float(v1[1])),
                            (float(v0[0]), float(v0[1])),
                            collision_m=collision,
                            ll_to_m=_ll_to_m_any,
                            m_to_ll=_m_to_ll_any,
                            repairer_obj=repairer_obj,
                        )
                    else:
                        inject_end_ll = [(float(v1[0]), float(v1[1])), (float(v0[0]), float(v0[1]))]

                    # ----------------------------
                    # (2) Repair CORE path only: path[1:-1]
                    #     This excludes inject edges completely.
                    # ----------------------------
                    core_nodes = path[1:-1]  # from u1 .. v1
                    if core_nodes is None or len(core_nodes) < 2:
                        core_repaired_ll = [(float(u1[0]), float(u1[1]))] if len(path) >= 2 else []
                        print("[core] skip repair: core_nodes<2")
                    else:
                        core_rep = repairer_obj.repair_path(
                            G,
                            core_nodes,
                            collision_m=collision,
                            ll_to_m=_ll_to_m_any,
                            m_to_ll=_m_to_ll_any,
                        )
                        print("[core-repair]", core_rep.stats)
                        core_repaired_ll = [(float(p[0]), float(p[1])) for p in core_rep.path_ll]

                    # ----------------------------
                    # (3) Stitch: inject_start + core + inject_end
                    # ----------------------------
                    def _extend_no_dup(dst, src):
                        if not src:
                            return
                        if not dst:
                            dst.extend(src)
                            return
                        if dst[-1] == src[0]:
                            dst.extend(src[1:])
                        else:
                            dst.extend(src)

                    full_ll = []
                    _extend_no_dup(full_ll, inject_start_ll)
                    _extend_no_dup(full_ll, core_repaired_ll)
                    _extend_no_dup(full_ll, inject_end_ll)

                    path_ll_for_simplify = full_ll
                    print("[collision used for debug] bounds:", collision.bounds, "type:", collision.geom_type)

                else:
                    path_ll_for_simplify = path_ll

                # simplify
                # simplify
                if do_simplify and path_ll_for_simplify is not None and len(path_ll_for_simplify) >= 2:
                    path_simplified, simp_stats = simplify_path_visibility(
                        path_ll_for_simplify,
                        collision_m=collision,
                        ll_to_m=_ll_to_m_any,
                        m_to_ll=_m_to_ll_any,
                        window_size=80,
                        max_tries=300,
                        use_prepared_collision=True,
                        dateline_unwrap=True,
                    )
                    print("[simplify]", simp_stats)

                    # ---------------------------
                    # 1) Decide CORE final polyline (priority: simplified > repaired_full > original A*)
                    # ---------------------------
                    core_final_ll = None
                    if path_simplified is not None and len(path_simplified) >= 2:
                        core_final_ll = [(float(p[0]), float(p[1])) for p in path_simplified]
                    elif path_ll_for_simplify is not None and len(path_ll_for_simplify) >= 2:
                        core_final_ll = [(float(p[0]), float(p[1])) for p in path_ll_for_simplify]
                    elif path is not None and len(path) >= 2:
                        core_final_ll = [(float(p[0]), float(p[1])) for p in path]

                    # ---------------------------
                    # 2) Build snap-links (origin->start_key, end_key->dest) and MERGE into FINAL
                    # ---------------------------
                    final_ll = core_final_ll

                    if core_final_ll is not None and len(core_final_ll) >= 2:
                        # snap-link start
                        if origin_ll != start_key:
                            snap_start_ll = repair_snap_link_ll_if_needed(
                                origin_ll, start_key,
                                collision_m=collision,
                                ll_to_m=_ll_to_m_any,
                                m_to_ll=_m_to_ll_any,
                                repairer_obj=repairer_obj,   # 你前面已經建過 PathRepairer 了，直接重用
                            )
                        else:
                            snap_start_ll = [origin_ll]

                        # snap-link end
                        if end_key != dest_ll:
                            snap_end_ll = repair_snap_link_ll_if_needed(
                                end_key, dest_ll,
                                collision_m=collision,
                                ll_to_m=_ll_to_m_any,
                                m_to_ll=_m_to_ll_any,
                                repairer_obj=repairer_obj,
                            )
                        else:
                            snap_end_ll = [dest_ll]

                        # merge without duplicate joints
                        merged = []
                        if snap_start_ll and len(snap_start_ll) >= 2:
                            merged.extend([(float(p[0]), float(p[1])) for p in snap_start_ll])
                        else:
                            merged.append((float(origin_ll[0]), float(origin_ll[1])))

                        # connect to core
                        if merged and core_final_ll:
                            if merged[-1] == core_final_ll[0]:
                                merged.extend(core_final_ll[1:])
                            else:
                                merged.extend(core_final_ll)

                        # connect end snap-link
                        if snap_end_ll and len(snap_end_ll) >= 2:
                            snap_end_ll = [(float(p[0]), float(p[1])) for p in snap_end_ll]
                            if merged and merged[-1] == snap_end_ll[0]:
                                merged.extend(snap_end_ll[1:])
                            else:
                                merged.extend(snap_end_ll)

                        final_ll = merged

                    # ---------------------------
                    # 3) Distance on FINAL (includes snap-links!)
                    # ---------------------------
                    if final_ll is not None and len(final_ll) >= 2:
                        total_km, total_nm = path_length_km_nm(final_ll, dateline_unwrap=True)
                        print(f"[distance] final route = {format_distance(total_km, total_nm)}")

                        try:
                            out["_p2p_last_distance"] = {"km": float(total_km), "nm": float(total_nm)}
                        except Exception:
                            pass
                    else:
                        print("[distance] skip: no final polyline")
                else:
                    print("[simplify] skip")
                

    # ---------------------------
    # folium map + layers
    # ---------------------------
    m = folium.Map(location=center, zoom_start=zoom_start, control_scale=True)
    folium.Rectangle(bounds=[[min_lat, min_lon], [max_lat, max_lon]], fill=False, weight=3, opacity=0.9).add_to(m)
    m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

    # draw collision used (optional)
    try:
        collision_viz = collision_used
        if collision_viz is not None:
            print("[collision viz ] bounds:", collision_viz.bounds, "type:", collision_viz.geom_type)
            ll2m_xy, m2ll_xy, _, _ = _make_ll_m_projectors_from_out(out)
            collision_viz = _clip_collision_to_aoi_bbox(collision_viz, bbox_ll, ll2m_xy, pad_m=50_000.0)
            col_ll = _geom_m_to_ll(collision_viz, m2ll_xy)
            from shapely.geometry import box as shp_box
            min_lon, min_lat, max_lon, max_lat = map(float, bbox_ll)
            # 建立一個完美的經緯度矩形
            viz_box = shp_box(min_lon, min_lat, max_lon, max_lat)
            # 只保留在這個矩形內的圖形
            col_ll = col_ll.intersection(viz_box)
            fgCol = folium.FeatureGroup(name="Collision (USED, approx ll)", show=False)
            folium.GeoJson(
                data=col_ll.__geo_interface__,
                style_function=lambda _: {"fillColor": "#3b82f6", "color": "#3b82f6", "weight": 2, "fillOpacity": 0.15},
            ).add_to(fgCol)
            fgCol.add_to(m)
    except Exception as e:
        print("[viz] collision layer failed:", repr(e))

    def circle_layer(df, name, radius, show=True):
        fg = folium.FeatureGroup(name=name, show=show)
        for _, r in df.iterrows():
            p = (float(r["lon"]), float(r["lat"]))
            if not in_bbox(p, bbox_ll):
                continue
            folium.CircleMarker([p[1], p[0]], radius=radius).add_to(fg)
        fg.add_to(m)

    # --- node layers ---
    if isinstance(C_nodes, pd.DataFrame) and len(C_nodes) > 0:
        nC = len(C_nodes)
        dfC_plot = C_nodes.sample(min(int(c_sample), nC), random_state=7) if nC > c_sample else C_nodes
        circle_layer(dfC_plot, f"C_nodes (sample {len(dfC_plot)}/{nC})", radius=1, show=False)

    if isinstance(dfGB, pd.DataFrame) and len(dfGB) > 0:
        circle_layer(dfGB, f"Gate_B ({len(dfGB)})", radius=6, show=False)

    if isinstance(S_nodes, pd.DataFrame) and len(S_nodes) > 0:
        nS = len(S_nodes)
        if s_sample is not None and nS > int(s_sample):
            dfS_plot = S_nodes.sample(int(s_sample), random_state=7)
            title = f"S_nodes (sample {len(dfS_plot)}/{nS})"
        else:
            dfS_plot = S_nodes
            title = f"S_nodes ({nS})"
        circle_layer(dfS_plot, title, radius=3, show=False)

    # --- sea edges (viz) ---
    def sea_lonlat_by_idx(i):
        s = S_nodes.iloc[int(i)]
        return (float(s["lon"]), float(s["lat"]))

    if isinstance(S_nodes, pd.DataFrame) and S_edges is not None and len(S_edges) > 0:
        fgE = folium.FeatureGroup(name=f"S_edges (show {min(len(S_edges), max_sea_edges_viz)}/{len(S_edges)})", show=True)
        take = S_edges[:max_sea_edges_viz] if len(S_edges) > max_sea_edges_viz else S_edges
        drawn = 0
        for e in take:
            seg = edge_to_lonlat(e, nodes_df=S_nodes, idx_to_lonlat_fn=sea_lonlat_by_idx)
            if seg is None:
                continue
            u, v = seg
            if not (in_bbox(u, bbox_ll) or in_bbox(v, bbox_ll)):
                continue
            folium.PolyLine([[u[1], u[0]], [v[1], v[0]]], color="#3352ff", weight=2, opacity=0.6).add_to(fgE)
            drawn += 1
        fgE.add_to(m)
        print(f"[viz] sea edges drawn: {drawn}/{len(take)}")

    # --- GateB→Sea connectors (viz) ---
    if isinstance(dfGB_conn, pd.DataFrame) and isinstance(S_nodes, pd.DataFrame) and gate_xy:
        fgConn = folium.FeatureGroup(name=f"GateB→Sea connectors ({len(dfGB_conn)})", show=False)
        drawn = 0
        for _, r in dfGB_conn.iterrows():
            try:
                gid = int(r["g_id"])
                if gid not in gate_xy:
                    continue
                gb = gate_xy[gid]
                if not in_bbox(gb, bbox_ll):
                    continue
                sea = sea_lonlat_by_idx(int(r["sea_idx"]))
                if not (in_bbox(gb, bbox_ll) or in_bbox(sea, bbox_ll)):
                    continue
                folium.PolyLine([[gb[1], gb[0]], [sea[1], sea[0]]], weight=2, opacity=0.7).add_to(fgConn)
                drawn += 1
            except Exception:
                continue
        fgConn.add_to(m)
        print(f"[viz] GateB→Sea connectors drawn: {drawn}/{len(dfGB_conn)}")

    # --- C↔GateB bridge connectors (viz) ---
    if isinstance(cgb_df, pd.DataFrame) and len(cgb_df) > 0:
        fgCGB = folium.FeatureGroup(name=f"C↔GateB bridge (nearest) ({len(cgb_df)})", show=False)
        for _, r in cgb_df.iterrows():
            c = (float(r["c_lon"]), float(r["c_lat"]))
            gb = (float(r["g_lon"]), float(r["g_lat"]))
            folium.PolyLine([[c[1], c[0]], [gb[1], gb[0]]], weight=2, opacity=0.7).add_to(fgCGB)
        fgCGB.add_to(m)

    # --- start/end markers + candidates ---
    fgSE = folium.FeatureGroup(name="Start/End + snapped candidates", show=True)
    folium.Marker([origin_ll[1], origin_ll[0]], tooltip="START (input)").add_to(fgSE)
    folium.Marker([dest_ll[1], dest_ll[0]], tooltip="END (input)").add_to(fgSE)

    if start_key != origin_ll:
        folium.CircleMarker([start_key[1], start_key[0]], radius=7, opacity=0.9, tooltip="START (used / nudged)").add_to(fgSE)
    if end_key != dest_ll:
        folium.CircleMarker([end_key[1], end_key[0]], radius=7, opacity=0.9, tooltip="END (used / nudged)").add_to(fgSE)

    for i, c in enumerate(pair.start_pick):
        folium.CircleMarker([c.node_ll[1], c.node_ll[0]], color="#6f42c1", radius=6, tooltip=f"start_cand#{i} d={c.dist_km:.1f}km").add_to(fgSE)
    for i, c in enumerate(pair.end_pick):
        folium.CircleMarker([c.node_ll[1], c.node_ll[0]], color="#6f42c1", radius=6, tooltip=f"end_cand#{i} d={c.dist_km:.1f}km").add_to(fgSE)
    fgSE.add_to(m)

    # --- route layers ---
    if path is not None and len(path) >= 2:
        fgA = folium.FeatureGroup(name="Route: A* (original core)", show=True)
        path_u = route_polyline_dateline_safe([(float(p[0]), float(p[1])) for p in path])
        folium.PolyLine([[p[1], p[0]] for p in path_u], color="#d62728", weight=6, opacity=0.95).add_to(fgA)
        fgA.add_to(m)

        #  Inject edges (the two edges you care about)
        has_inj_start = (inject_start_ll is not None and len(inject_start_ll) >= 2 and inject_start_ll[0] != inject_start_ll[-1])
        has_inj_end   = (inject_end_ll   is not None and len(inject_end_ll)   >= 2 and inject_end_ll[0]   != inject_end_ll[-1])
        if has_inj_start or has_inj_end:
            fgSL = folium.FeatureGroup(name="Route: Inject edges (repaired if needed)", show=True)
            if has_inj_start:
                sl1 = route_polyline_dateline_safe([(float(p[0]), float(p[1])) for p in inject_start_ll])
                folium.PolyLine([[p[1], p[0]] for p in sl1], color="#9467bd", weight=5, opacity=0.92).add_to(fgSL)
            if has_inj_end:
                sl2 = route_polyline_dateline_safe([(float(p[0]), float(p[1])) for p in inject_end_ll])
                folium.PolyLine([[p[1], p[0]] for p in sl2], color="#9467bd", weight=5, opacity=0.92).add_to(fgSL)
            fgSL.add_to(m)

        # repaired full (pre-simplify)
        if path_ll_for_simplify is not None and len(path_ll_for_simplify) >= 2:
            fgR = folium.FeatureGroup(name="Route: Repaired (FULL, pre-simplify)", show=True)
            repaired_ll = [(float(p[0]), float(p[1])) for p in path_ll_for_simplify]
            path_ru = route_polyline_dateline_safe(repaired_ll)
            folium.PolyLine([[p[1], p[0]] for p in path_ru], color="#2ca02c", weight=5, opacity=0.90).add_to(fgR)
            fgR.add_to(m)

        # simplified
        if path_simplified is not None and len(path_simplified) >= 2:
            fgS = folium.FeatureGroup(name="Route: Simplified (visibility)", show=True)
            path_su = route_polyline_dateline_safe([(float(p[0]), float(p[1])) for p in path_simplified])
            folium.PolyLine([[p[1], p[0]] for p in path_su], color="#ff7f0e", weight=5, opacity=0.95).add_to(fgS)
            fgS.add_to(m)
        
        # --- FINAL route (with snap-links) ---
        if final_ll is not None and len(final_ll) >= 2:
            fgF = folium.FeatureGroup(name="Route: FINAL (simplified + snap-links)", show=True)
            final_u = route_polyline_dateline_safe(final_ll)
            folium.PolyLine([[p[1], p[0]] for p in final_u], weight=6, opacity=0.95).add_to(fgF)
            fgF.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    html_path = Path(html_path).resolve()
    m.save(str(html_path))
    webbrowser.open(html_path.as_uri())
    return html_path


# =========================
# === call
# =========================
origin_ll = (128.52636, -30.38197)
dest_ll   = (135.134173, 34.200334)

open_routing_debug_map_p2p(
    out,
    origin_ll=origin_ll,
    dest_ll=dest_ll,
    html_path="aoi_p2p_map.html",
    zoom_start=5,
    include_sea=True,
    include_cc=True,
    include_gateb_sea=True,
    use_c_gateb_bridge=True,
    c_to_gateB_max_deg_dist=None,
    k_near=30,
    r_max_km_snap=150,
    k_inject=4,
    do_repair=True,
    do_simplify=True,
)


[graph] base stats: GraphBuildStats(sea_edges_added=4813, cc_edges_added=9710, gateb_sea_edges_added=264, c_gateb_edges_added=0)
[graph] C↔GateB bridge edges added: 119
[snap] OK: common_component_preferred {'start_reason': 'ok_within_radius_fallback', 'end_reason': 'ok_within_radius', 'start_mode': 'coast_then_sea', 'end_mode': 'coast_then_sea', 'start_fallback': 'coastal_nudge_failed', 'end_fallback': None, 'common_components': [0], 'chosen_common_component': 0, 'largest_component': 0}
[snap][start] local_entrance_aug: {'enabled': True, 'triggered': True, 'reason': 'dist>60km,end_gap>120km,angle>110deg', 'target_ll': (135.134173, 34.200334), 'd_end_seed0_km': 7399.255820339968, 'd_end_min_topK_km': 6233.1459252332215, 'd_end_gap_km': 1166.109895106746, 'angle_diff_deg': 158.18153758837877, 'virtual_added': 6, 'cands_all_after': 36}
[snap][end  ] local_entrance_aug: {'enabled': True, 'triggered': True, 'reason': 'end_gap>120km,angle>110deg', 'target_ll': (128.52636, -30.38197), 'd_end

WindowsPath('C:/Users/slab/Desktop/Slab Project/Stage2 ETA/aoi_p2p_map.html')

#### debug

##### 碰撞幾何